# 05 - Combine models (ensemble)

**Goal:** mix the predictions of several models. Different models make different mistakes, so a mix is often better than any one.

Three ways, all checked on the out-of-fold predictions:
1. **Simple average** of the probabilities.
2. **Rank average:** turn each model's predictions into ranks first. AUC only cares about order, so this is safe and often a little better.
3. **Stacking:** a small logistic regression learns how much to trust each model. It is trained with the same 5 folds, so its score is honest.

In [1]:
import sys
from pathlib import Path
sys.path.insert(0, str(Path.cwd().parent / "src"))

import numpy as np
import pandas as pd
import warnings
warnings.filterwarnings("ignore")

from config import *
from utils import *
from scipy.stats import rankdata
from sklearn.linear_model import LogisticRegression

In [2]:
train, test, sample = load_data()
folds = get_folds(train)
y = train[TARGET].values

exp = pd.read_csv(EXPERIMENTS_FILE).sort_values("cv_auc", ascending=False)
names = [n for n in exp["model"] if (OOF_DIR / f"{n}_oof.npy").exists()]
OOF = pd.DataFrame({n: load_preds(n)[0] for n in names})
TEST = pd.DataFrame({n: load_preds(n)[1] for n in names})
assert len(OOF) == len(train) and len(TEST) == len(test)
exp

,model,features,cv_auc,cv_logloss,notes,date
10,lgbm_orig,raw+orig way B,0.942877,0.224594,"original dataset, way B, lr 0.05",2026-10-17 18:07
8,blend_rank,lgbm_optuna+lgbm_fe,0.942846,NaN,ensemble,2026-10-17 16:42
7,blend_mean,lgbm_optuna+lgbm_fe,0.942844,NaN,ensemble,2026-10-17 16:42
9,blend_stack,lgbm_optuna+lgbm_fe,0.942842,0.224661,ensemble,2026-10-17 16:42
4,lgbm_optuna,raw,0.942828,0.224683,"optuna 46 trials, lr 0.02",2026-10-17 15:59
5,lgbm_fe,raw+A,0.942806,0.224730,lgbm with kept feature groups,2026-10-17 16:12
2,xgb_v1,raw,0.941857,0.226431,default params,2026-10-17 15:11
6,cat_fe,raw+A,0.941778,0.226582,catboost with kept feature groups,2026-10-17 16:39
3,cat_v1,raw,0.941772,0.226594,default params,2026-10-17 15:39
1,lgbm_v1,raw,0.941771,0.226571,default params,2026-10-17 15:09


How alike are the models? (1.0 = identical predictions. Lower means they add more variety to a blend.)

In [3]:
OOF.corr(method="spearman").round(4)

,lgbm_orig,lgbm_optuna,lgbm_fe,xgb_v1,cat_fe,cat_v1,lgbm_v1,logreg_v1
lgbm_orig,1.0000,0.9976,0.9975,0.9955,0.9942,0.9941,0.9956,0.9863
lgbm_optuna,0.9976,1.0000,0.9992,0.9960,0.9949,0.9948,0.9955,0.9859
lgbm_fe,0.9975,0.9992,1.0000,0.9960,0.9947,0.9947,0.9954,0.9855
xgb_v1,0.9955,0.9960,0.9960,1.0000,0.9957,0.9957,0.9969,0.9864
cat_fe,0.9942,0.9949,0.9947,0.9957,1.0000,0.9991,0.9948,0.9861
cat_v1,0.9941,0.9948,0.9947,0.9957,0.9991,1.0000,0.9947,0.9860
lgbm_v1,0.9956,0.9955,0.9954,0.9969,0.9948,0.9947,1.0000,0.9890
logreg_v1,0.9863,0.9859,0.9855,0.9864,0.9861,0.9860,0.9890,1.0000


## Pick models with greedy forward selection
Start with the best model. Keep adding whichever model raises the rank-average AUC the most. Stop when nothing helps.

In [4]:
def rank(df):
    return df.apply(lambda c: rankdata(c) / len(c))

R_OOF, R_TEST = rank(OOF), rank(TEST)

chosen = [names[0]]
best = score(y, R_OOF[chosen].mean(axis=1))[0]
print(f"start: {chosen[0]}  AUC = {best:.5f}")
while True:
    tries = {n: score(y, R_OOF[chosen + [n]].mean(axis=1))[0] for n in names if n not in chosen}
    if not tries:
        break
    n, a = max(tries.items(), key=lambda kv: kv[1])
    if a <= best:
        break
    chosen.append(n); best = a
    print(f"add:   {n:<14} AUC = {best:.5f}")
print("Chosen models:", chosen)

start: lgbm_orig  AUC = 0.94288


add:   lgbm_optuna    AUC = 0.94299


Chosen models: ['lgbm_orig', 'lgbm_optuna']


In [5]:
blends = {}

# 1. simple average
p = OOF[chosen].mean(axis=1)
blends["blend_mean"] = (score(y, p)[0], TEST[chosen].mean(axis=1))

# 2. rank average
p = R_OOF[chosen].mean(axis=1)
blends["blend_rank"] = (score(y, p)[0], R_TEST[chosen].mean(axis=1))

# 3. stacking with logistic regression on log-odds of each model, same folds
def logit(p):
    p = np.clip(p, 1e-6, 1 - 1e-6); return np.log(p / (1 - p))

Z, Z_test = logit(OOF[chosen].values), logit(TEST[chosen].values)
stack_oof = np.zeros(len(y))
for k in range(N_FOLDS):
    tr, va = folds != k, folds == k
    stack_oof[va] = LogisticRegression(C=1.0).fit(Z[tr], y[tr]).predict_proba(Z[va])[:, 1]
stacker = LogisticRegression(C=1.0).fit(Z, y)
print("Stacking weights:", dict(zip(chosen, stacker.coef_[0].round(3))))
blends["blend_stack"] = (score(y, stack_oof)[0], stacker.predict_proba(Z_test)[:, 1])

for n, (a, _) in blends.items():
    print(f"{n:<12} CV AUC = {a:.5f}")
print(f"{'best single':<12} CV AUC = {exp['cv_auc'].iloc[0]:.5f}  ({names[0]})")

Stacking weights: {'lgbm_orig': np.float64(0.5), 'lgbm_optuna': np.float64(0.506)}


blend_mean   CV AUC = 0.94299
blend_rank   CV AUC = 0.94299
blend_stack  CV AUC = 0.94299
best single  CV AUC = 0.94288  (lgbm_orig)


## Save the blends
Note: the rank blend gives ranks between 0 and 1, not true probabilities. That is fine for AUC.
If the metric turns out to be logloss, use `blend_stack` or `blend_mean` instead.

In [6]:
for n, (a, t) in blends.items():
    make_submission(np.asarray(t), n, sample)
    ll = score(y, stack_oof)[1] if n == "blend_stack" else np.nan
    log_experiment(n, "+".join(chosen), a, ll, "ensemble")

final = pd.read_csv(EXPERIMENTS_FILE).sort_values("cv_auc", ascending=False)
final

Saved blend_mean.csv  rows=286571


Saved blend_rank.csv  rows=286571


Saved blend_stack.csv  rows=286571


,model,features,cv_auc,cv_logloss,notes,date
9,blend_rank,lgbm_orig+lgbm_optuna,0.942989,NaN,ensemble,2026-10-17 18:07
8,blend_mean,lgbm_orig+lgbm_optuna,0.942988,NaN,ensemble,2026-10-17 18:07
10,blend_stack,lgbm_orig+lgbm_optuna,0.942986,0.224397,ensemble,2026-10-17 18:07
7,lgbm_orig,raw+orig way B,0.942877,0.224594,"original dataset, way B, lr 0.05",2026-10-17 18:07
4,lgbm_optuna,raw,0.942828,0.224683,"optuna 46 trials, lr 0.02",2026-10-17 15:59
5,lgbm_fe,raw+A,0.942806,0.224730,lgbm with kept feature groups,2026-10-17 16:12
2,xgb_v1,raw,0.941857,0.226431,default params,2026-10-17 15:11
6,cat_fe,raw+A,0.941778,0.226582,catboost with kept feature groups,2026-10-17 16:39
3,cat_v1,raw,0.941772,0.226594,default params,2026-10-17 15:39
1,lgbm_v1,raw,0.941771,0.226571,default params,2026-10-17 15:09


## Final pick (Kaggle lets you choose 2)
1. The submission with the **highest CV AUC that holds real probabilities**.
   `blend_rank` is skipped because its values are ranks. They would score badly if the metric turns out to be logloss.
2. A safer second choice: a **different type** of submission (e.g. the best single model if #1 is a blend).

In [7]:
safe = final[final["model"] != "blend_rank"]
top = safe.iloc[0]["model"]
second = safe[safe["notes"] != safe.iloc[0]["notes"]].iloc[0]["model"]
print("Final pick 1:", f"submissions/{top}.csv")
print("Final pick 2:", f"submissions/{second}.csv")

Final pick 1: submissions/blend_mean.csv
Final pick 2: submissions/lgbm_orig.csv
